In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import colorir as cl
from plotly.subplots import make_subplots
from analyses import io

In [ ]:
colors = cl.StackPalette.load("safe")
colors

In [ ]:
dfs = [
    io.read_celldfs("../runs/pseudopodia1/", low_memory=True), 
    io.read_celldfs("../runs/pseudopodia2/", low_memory=True), 
    io.read_celldfs("../runs/pseudopodia3/", low_memory=True)
]

In [ ]:
celldf = pl.concat(dfs).with_columns(
    replica=pl.col("replica").cast(pl.UInt32),
    gamma=20 - pl.col("energy").str.split("-").list.get(-1).cast(pl.UInt32),
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5
).with_columns(
    mean_displ=pl.col("displ").mean().over("gamma", "replica", "time")
).drop("energy")
celldf

In [ ]:
max_time = celldf["time"].max()
min_steady_time = 3e6
stdf = celldf\
    .group_by(["gamma", "replica"])\
    .agg(
        stime=pl.col("time")\
            .filter(
                pl.col("mean_displ") <= pl.col("mean_displ").filter(pl.col("time") > min_steady_time).mean()
            )\
            .min(),
        ptime=pl.col("time")\
            .filter(
                pl.col("mean_displ") <= 180
            )\
            .min().fill_null(max_time)
    )
stdf

In [ ]:
kactdf = celldf.join(
    stdf, 
    on=["replica", "gamma"]
).filter(
    pl.col("time") > 5e5,
    pl.col("time") < pl.col("stime") * 0.9,
    pl.col("med_neighbor") == True,
).with_columns(
    mean_x=pl.col("center_x").mean().over("gamma", "replica", "time"),
    mean_y=pl.col("center_y").mean().over("gamma", "replica", "time")
).with_columns(
    dx=pl.col("kact_center_x") - pl.col("center_x"),
    dy=pl.col("kact_center_y") - pl.col("center_y"),
    cell_dx=pl.col("center_x") - pl.col("mean_x"),
    cell_dy=pl.col("center_y") - pl.col("mean_y"),
).with_columns(
    # Angle of pseudopodium relative to cell center 
    angle=(pl.arctan2(pl.col("dy"), pl.col("dx")).degrees() + 135) % 360,
    # Angle of cell center relative to cluster center
    cell_angle=(pl.arctan2(pl.col("cell_dy"), pl.col("cell_dx")).degrees() + 135) % 360,
    mag=(pl.col("dx") ** 2 + pl.col("dy") ** 2) ** 0.5,
    cell_mag=(pl.col("cell_dx") ** 2 + pl.col("cell_dy") ** 2) ** 0.5,
).with_columns(
    angle_bin=pl.col("cell_angle").cut(np.linspace(0, 360, 16), include_breaks=True).struct.field("breakpoint"),
    cos_sim=(pl.col("angle") - pl.col("cell_angle")).radians().cos(),
    front=(pl.col("cell_angle") < 30) | (pl.col("cell_angle") >= 330),
    back=(pl.col("cell_angle") > 150) & (pl.col("cell_angle") <= 210),
).with_columns(
    pull_strength=pl.col("cos_sim") * pl.col("tot_kact"),
    ux=pl.col("dx") / pl.col("mag"),
    uy=pl.col("dy") / pl.col("mag"),
    cell_ux=pl.col("cell_dx") / pl.col("mag"),
    cell_uy=pl.col("cell_dy") / pl.col("mag")
).drop_nans()  # When dx = 0 and dy = 0 there cant be an angle
kactdf

In [ ]:
angledf = kactdf.group_by("gamma", "angle_bin").agg(
    pl.col("cos_sim").mean(),
    pl.col("pull_strength").mean(),
    ux=pl.col("ux").sum(),
    uy=pl.col("uy").sum(),    
    cell_ux=pl.col("cell_ux").sum(),
    cell_uy=pl.col("cell_uy").sum(),    
    mean_kact=pl.col("tot_kact").mean()
).with_columns(
    # We first average the bins and then take the cos_sim, otherwise correlation is poor within each individual cell...
    # This is approx uniform around the circle
    cos_sim_b=(pl.arctan2("uy", "ux") - pl.arctan2("cell_uy", "cell_ux")).cos(),
).with_columns(
    # This aint (bc tot_kact isnt)
    pull_strength_b=pl.col("mean_kact") * pl.col("cos_sim_b")
).sort(
    "gamma"
)
angledf

In [ ]:
px.strip(
    angledf.sort("angle_bin"),
    x="gamma",
    y="cos_sim",
    color="angle_bin",
    color_discrete_sequence=cl.StackPalette.load("twilight").resize(16)
).update_traces(
    jitter=1
).update_layout(
    width=500,
    height=300,
    template="plotly_white"
)

In [ ]:
for (gamma,), filterdf in kactdf.sort("gamma").group_by("gamma", maintain_order=True):
    bindf = angledf.filter(
        pl.col("gamma") == gamma
    )
    fig = px.bar_polar(
        bindf,
        r="pull_strength",
        theta="angle_bin",
        direction="counterclockwise"
    ).update_traces(
        marker_color=colors[0]
    )
    
    tip = filterdf.filter(front=True)["pull_strength"].mean() - filterdf.filter(front=False)["pull_strength"].mean()
    angle = (np.degrees(np.arctan2(
        (filterdf["cell_ux"] * filterdf["pull_strength"]).sum() / filterdf["pull_strength"].sum(), 
        (filterdf["cell_uy"] * filterdf["pull_strength"]).sum() / filterdf["pull_strength"].sum(), 
    )) + 135) % 360

    # This needs to be changed if not using range = 0 - ...
    zero = 0
    fig.add_trace(go.Scatterpolar(
        r=[zero, tip],
        theta=[zero, angle],
        mode="lines",
        line=dict(color="#fdca09", width=3),
        showlegend=False
    ))
    # arrowhead at the tip
    fig.add_trace(go.Scatterpolar(
        r=[zero, tip],
        theta=[zero, angle],
        mode="markers",
        marker=dict(
            symbol="triangle-up",
            size=10,
            color="#fdca09",
            angleref="previous",  # rotates marker to match line direction
        ),
        showlegend=False,
    ))
    
    for r in np.arange(0, bindf["pull_strength"].max(), 5):  # your desired grid radii
        theta = np.linspace(0, 360, 100)
        fig.add_trace(go.Scatterpolar(
            r=[r]*100,
            theta=theta,
            mode='lines',
            line=dict(color='darkgrey', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))
    fig.update_layout(
        width=300,
        height=300,
        template="plotly_white",
        title=gamma,
        polar_angularaxis_dtick=45,
        polar_radialaxis=dict(
            showgrid=False,
            dtick=5,
            range=[0, bindf["pull_strength"].max() * 1.1]
        )
    )
    fig.show()
    # io.save_plot(fig, f"../plots/run_plots/pull_strength_gamma-{gamma}")